## Import Libraries and Data Loading

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [3]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)

In [4]:
df = pd.read_csv('bank_marketing_training.csv')

## Data Audit

In [5]:
# Inspect column nams and data types (dtypes)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26874 entries, 0 to 26873
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  26874 non-null  int64  
 1   job                  26874 non-null  str    
 2   marital              26874 non-null  str    
 3   education            26874 non-null  str    
 4   default              26874 non-null  str    
 5   housing              26874 non-null  str    
 6   loan                 26874 non-null  str    
 7   contact              26874 non-null  str    
 8   month                26874 non-null  str    
 9   day_of_week          26874 non-null  str    
 10  duration             26874 non-null  int64  
 11  campaign             26874 non-null  int64  
 12  days_since_previous  26874 non-null  int64  
 13  previous             26874 non-null  int64  
 14  previous_outcome     26874 non-null  str    
 15  emp.var.rate         26874 non-null  float64
 1

In [6]:
# Quick summary of numerical columns
df.describe()

,age,duration,campaign,days_since_previous,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,26874.000000,26874.000000,26874.000000,26874.000000,26874.000000,26874.000000,26874.000000,26874.000000,26874.000000,26874.000000
mean,39.987758,257.741907,2.563035,962.714631,0.171058,0.086552,93.575794,-40.481670,3.627879,5167.103446
std,10.400285,258.977140,2.748746,186.324546,0.487657,1.569716,0.578933,4.626506,1.731948,72.080534
min,17.000000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.000000
25%,32.000000,103.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.000000
50%,38.000000,180.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.000000,319.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.000000
max,91.000000,4918.000000,43.000000,999.000000,6.000000,1.400000,94.767000,-26.900000,5.045000,5228.000000


In [7]:
# Separate numerical and categorical columns
cat_cols = df.select_dtypes(include=['str']).columns.tolist()
num_cols = df.select_dtypes(include=['number']).columns.tolist()

In [8]:
# Inspect categorical and numerical distribution
print(f"There are {len(cat_cols)} categorical columns, which are:\n", cat_cols)
print(f"There are {len(num_cols)} numerical columns, which are:\n", num_cols)

There are 11 categorical columns, which are:
 ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'previous_outcome', 'response']
There are 10 numerical columns, which are:
 ['age', 'duration', 'campaign', 'days_since_previous', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']


In [9]:
# Check missing values
df.isnull().sum().sort_values(ascending=False)

age                    0
job                    0
marital                0
education              0
default                0
housing                0
loan                   0
contact                0
month                  0
day_of_week            0
duration               0
campaign               0
days_since_previous    0
previous               0
previous_outcome       0
emp.var.rate           0
cons.price.idx         0
cons.conf.idx          0
euribor3m              0
nr.employed            0
response               0
dtype: int64

## Basic Data Cleaning

In [11]:
# Drop duplicate rows
df = df.drop_duplicates()

In [12]:
# Trim whitespace in categorical columns
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

In [13]:
# Detect outliers in numerical columns
iqr_mult = 1.5
print("Number of outliers")
for col in num_cols:
    if col in df.columns:
        q1, q3 = df[col].quantile([0.25,0.75])
        iqr = q3 - q1
        lower, upper = q1 - iqr * iqr_mult, q3 + iqr * iqr_mult
        outliers = len(df[(df[col]<lower) | (df[col]>upper)])
        print(f"{col}: {outliers}")

Number of outliers
age: 299
duration: 1898
campaign: 1578
days_since_previous: 982
previous: 3664
emp.var.rate: 0
cons.price.idx: 0
cons.conf.idx: 287
euribor3m: 0
nr.employed: 0


## Define Target and Features

In [22]:
# Choose target variable
target_col = 'marital'
assert target_col in df.columns, "Update target_col to match the dataset."

# Sample class balance
if target_col in df.columns:
    print(df[target_col].value_counts(normalize=True))

marital
married     0.602307
single      0.281876
divorced    0.113696
unknown     0.002121
Name: proportion, dtype: float64


In [25]:
X = df.drop(columns=[target_col])
y = df[target_col]

In [26]:
X.head()

,age,job,education,default,housing,loan,contact,month,day_of_week,duration,campaign,days_since_previous,previous,previous_outcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,response
0,56,housemaid,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191,no
1,57,services,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191,no
2,41,blue-collar,unknown,unknown,no,no,telephone,may,mon,217,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191,no
3,25,services,high.school,no,yes,no,telephone,may,mon,222,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191,no
4,29,blue-collar,high.school,no,no,yes,telephone,may,mon,137,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191,no


In [27]:
y.head()

0    married
1    married
2    married
3     single
4     single
Name: marital, dtype: str

## Encode categorical variables and impute missing values